<a href="https://colab.research.google.com/github/konovalovae812-cmd/python_da_hw/blob/main/lisa_hw_10_4_apply%2C_groupby%2C_pivot_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Домашнє завдання до тем apply(), groupby(), pivot_table()

В цьому домашньому завданні продовжуємо працювати з набором даних `supermarket_sales.csv`.

0. Імпортуйте бібліотеку pandas та зчитайте дані у змінну `df` типу `pandas.DataFrame`.

In [28]:
import pandas as pd
df = pd.read_csv('/content/supermarket_sales.csv')
print(type(df))

<class 'pandas.core.frame.DataFrame'>


1. Дослідимо, який філіал супермаркету ('Branch') є найприбутковішим. Для цього знайдіть сумарний прибуток за кожним філіалом і виявіть, який філіал має найвищий.

In [29]:
branch_profit = df.groupby('Branch')['gross income'].sum()
best_branch = branch_profit.idxmax()
max_profit = branch_profit.max()
print("Сумарний прибуток за філіалами:")
print(branch_profit)
print(f"\nНайприбутковіший філіал: {best_branch} з прибутком {max_profit:.2f}")

Сумарний прибуток за філіалами:
Branch
A    5057.1605
B    5057.0320
C    5265.1765
Name: gross income, dtype: float64

Найприбутковіший філіал: C з прибутком 5265.18


2. В якому місті знайходиться філіал з найвищим прибутком? Може в тому місці нам розмітисти ще один магазин.  
Знайдіть відповідь за допомогою функціоналу Pandas.

In [30]:
city_branch_profit = df.groupby(['City', 'Branch'])['gross income'].sum()
best_location = city_branch_profit.idxmax()
max_value = city_branch_profit.max()
print(f"Найприбутковіша локація: місто {best_location[0]}, філіал {best_location[1]}")
print(f"Сумарний прибуток: {max_value:.2f}")

Найприбутковіша локація: місто Naypyitaw, філіал C
Сумарний прибуток: 5265.18


3.1. Створіть зводну таблицю, яка покаже, скільки покупок (інвойсів) було зроблено в кожній з філій (`Branch`) за різними категоріями товарів. Запишіть таблицю в змінну `invoices_by_category` і виведіть змінну на екран.
Ця таблиця допоможе проаналізувати, в якій філії купують найбільше товарів кожної з категорій.

In [46]:
invoices_by_category = pd.pivot_table(
    df,
    index='Branch',
    columns='Product line',
    values='Invoice ID',
    aggfunc='count'
)
invoices_by_category

Product line,Electronic accessories,Fashion accessories,Food and beverages,Health and beauty,Home and lifestyle,Sports and travel
Branch,,,,,,
A,60,51,58,47,65,59
B,55,62,50,53,50,62
C,55,65,66,52,45,45


Очікуваний результат:

![](https://drive.google.com/uc?export=view&id=1rueAdko6S3UxIHGtojetTxlES-EyM6Yb)

3.2. Викристовуючи змінну `invoices_by_category` дайте відповідь програмно (тобто значення треба не просто знайти очима, а вивести за допомогою коду), в якому філіалі магазину (`Branch`) найбільше інвойсів із покупкою товарів категорії "Електронні аксесуари" (`Electronic accessories`)?


In [42]:
best_branch = invoices_by_category['Electronic accessories'].idxmax()

max_value = invoices_by_category['Electronic accessories'].max()

print(f"Філіал з найбільшою кількістю покупок електроніки: {best_branch}")
print(f"Кількість інвойсів: {max_value}")


Філіал з найбільшою кількістю покупок електроніки: A
Кількість інвойсів: 60


4-6. **Творче завдання на розвиток аналітичного мислення**

Крок 1. Сформулюйте ТРИ питання (гіпотези) до наявних даних, які допомогли б вам зрозуміти, які користувачі що, де та коли найбільше/найменше купують, аби дати на основі цих гіпотез рекомендації бізнесу. Звісно питання мають бути не тими, на які ми вже відповіли в завданнях модулю.

Крок 2. Знайдіть відповіді на свої питання з допомогою функціоналу pandas.

Крок 3. Напишіть, як відповідь на це питання може бути використана для прийняття бізнес рішень.   
   
 Питання можуть бути будь-якої складності, але їх має бути 3. Кожне питання оцінюється як 1 завдання. Без виконання цього завдання ДЗ не приймається. Якщо є питання щодо виконання - пишіть у чат 🙌


In [49]:
# 1 Гіпотеза. Чи відрізняється середній чек між чоловіками та жінками залежно від дня тижня (weekday vs weekend)?

df['Date'] = pd.to_datetime(df['Date'])
df['DayType'] = df['Date'].dt.dayofweek.apply(
    lambda x: 'Weekend' if x >= 5 else 'Weekday'
)

df.groupby(['Gender','DayType'])['Total'].mean()

# Жінки мають вищий середній чек. Обидві категорії мають найвищий середній чек на вихідних.
# Користуємось цією інформацією для планування рекламних акцій


Gender  DayType
Female  Weekday    332.714499
        Weekend    340.461000
Male    Weekday    300.382348
        Weekend    336.697259
Name: Total, dtype: float64

In [51]:
# 2 Гіпотеза. У які години в кожному місті спостерігається мінімальний оборот — і чи співпадають ці години між містами?

df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M').dt.hour

df.groupby(['City','Hour'])['Total'].sum() \
  .reset_index() \
  .sort_values(['City','Total']) \
  .groupby('City').head(1)

#У кожному місті свій 'невдалий час'. Використовуємо ці результати для палунвання промо-акцій, 'щасливих годин', графіку роботи кас тощо.

,City,Hour,Total
6,Mandalay,16,4123.5600
18,Naypyitaw,17,7560.4305
32,Yangon,20,5896.3800


In [52]:
# 3 Гіпотеза. Чи впливає спосіб оплати (Cash / Credit card / Ewallet) на розмір кошика та середній чек?
df.groupby('Payment').agg(
    avg_qty=('Quantity','mean'),
    avg_total=('Total','mean'),
    transactions=('Invoice ID','count')
)
# Cash - найвищий чек, ewallet - найбільше транзакцій, але найнижчий середній чек.
# Потрібно стимулювати канал ewallet для підвищення середнього чеку.
# Ввести спецільні пропозиції для Credit card

,avg_qty,avg_total,transactions
Payment,,,
Cash,5.511628,326.181890,344
Credit card,5.536977,324.009878,311
Ewallet,5.484058,318.820600,345
